In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------------------------------
# Paths
# -------------------------------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "FIFA_2017_2022_MERGED_CLEAN.csv"

# -------------------------------------------------
# FIFA files to merge
# -------------------------------------------------
FILES = {
    2017: "fifa_2017.csv",
    2018: "fifa_2018.csv",
    2019: "fifa_2019.csv",
    2020: "fifa_2020.csv",
    2021: "fifa_2021.csv",
    2022: "fifa_2022.csv",
}

# -------------------------------------------------
# Columns we actually care about (project-driven)
# -------------------------------------------------
KEEP_COLUMNS = [
    "Name", "Nationality", "Club", "Position",
    "Overall", "Potential",
    "Acceleration", "SprintSpeed",
    "Finishing", "ShotPower", "LongShots", "Volleys", "Penalties",
    "ShortPassing", "LongPassing", "Vision", "Crossing", "Curve", "FKAccuracy",
    "Interceptions", "Marking", "StandingTackle", "SlidingTackle",
    "Strength", "Stamina", "Aggression", "Jumping", "Balance",
    "Dribbling"
]

# -------------------------------------------------
# Helper functions
# -------------------------------------------------
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = (
                df[c]
                .astype(str)
                .str.replace(",", "", regex=False)
                .replace({"": np.nan, "nan": np.nan})
            )
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# -------------------------------------------------
# Load, clean, and append each year
# -------------------------------------------------
frames = []

for year, file_name in FILES.items():
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    df = pd.read_csv(file_path)
    df = clean_columns(df)

    # Add year column
    df["fifa_year"] = year

    # Keep only relevant columns
    available_cols = [c for c in KEEP_COLUMNS if c in df.columns]
    df = df[available_cols + ["fifa_year"]]

    frames.append(df)

# -------------------------------------------------
# Merge all years
# -------------------------------------------------
fifa = pd.concat(frames, ignore_index=True)

# -------------------------------------------------
# Convert numeric fields
# -------------------------------------------------
NUMERIC_COLS = [c for c in KEEP_COLUMNS if c not in ["Name", "Nationality", "Club", "Position"]]
fifa = to_numeric(fifa, NUMERIC_COLS)

# -------------------------------------------------
# Basic cleaning
# -------------------------------------------------
# Drop rows missing core ratings
fifa = fifa.dropna(subset=["Overall", "Potential"])

# Valid rating range
fifa = fifa[fifa["Overall"].between(1, 100)]
fifa = fifa[fifa["Potential"].between(1, 100)]

# Remove duplicates (player-year)
fifa = fifa.drop_duplicates(subset=["Name", "fifa_year"])

# -------------------------------------------------
# Save clean merged file
# -------------------------------------------------
fifa.to_csv(OUTPUT_FILE, index=False)

print("✅ Clean merged dataset saved to:", OUTPUT_FILE)
print("Rows:", len(fifa))
print("Columns:", len(fifa.columns))